In [ ]:
from aistudio_common.utils.notebook_utils import NotebookUtils
from aistudio_common.utils import env_utils
from aistudio_notebook.utils.notebook_extension_util import topai, get_targets, get_args

'''
notebook 默认初始化代码，导入常用工具包，定义df1、df2、df3、df4、o，若不使用可删除该cell
'''
o = env_utils.get_odps_instance()
args = get_args()
targets = get_targets()
df1 = NotebookUtils.get_df(1)
df = df1.to_pandas()
print(len(df))

In [ ]:
import pandas as pd

# 假设 df 是你的 DataFrame
# 1. 预计算 Level 3 的映射 (code1, code2, code3) -> 对应的 id 集合
# 使用 set 主要是为了利用集合减法的高效性
g3 = df.groupby(['code1', 'code2', 'code3'])['benefit_id'].apply(set).reset_index()
g3.rename(columns={'benefit_id': 'set_sim3'}, inplace=True)

# 2. 预计算 Level 2 的映射 (code1, code2) -> 对应的 id 集合
g2 = df.groupby(['code1', 'code2'])['benefit_id'].apply(set).reset_index()
g2.rename(columns={'benefit_id': 'set_sim2'}, inplace=True)

# 3. 将预计算结果拼回原表 (这一步非常快，是向量化的)
df_opt = df.merge(g3, on=['code1', 'code2', 'code3'], how='left')
df_opt = df_opt.merge(g2, on=['code1', 'code2'], how='left')

# 4. 使用 zip 和 列表推导式 进行最终计算
# 比 df.apply 还要快，因为避免了 Pandas Series 的构建开销
item_id = []
sim_items = []

# 提取出 numpy array 进行迭代，速度飞快
iterator = zip(
    df_opt['benefit_id'], 
    df_opt['set_sim3'], 
    df_opt['set_sim2']
)
index = 0
for current_id, set3, set2 in iterator:
    item_id.append(current_id)
    
    # --- 逻辑优化区 ---
    
    # 逻辑1：Sim3 是同组内排除自己
    # set3 是所有匹配 code1,2,3 的集合
    final_sim3 = [f"{x}:3" for x in set3 if x != current_id]
    
    # 逻辑2：Sim2 是匹配 code1,2 的，但要排除掉已经在 Sim3 里的
    # 原代码逻辑：Sim2_list - self - Sim3_list
    # 数学等价于：Set2 - Set3 (因为 Set3 包含了 self，所以 self 也会被顺便减掉)
    diff_set = set2 - set3
    final_sim2 = [f"{x}:2" for x in diff_set]
    
    # 合并结果
    sim_items.append(final_sim3 + final_sim2)
    index += 1
    if index % 1000 == 0:
        print(index)
    
# 此时 item_id 和 sim_items 就是你要的结果
print("Done!")

In [ ]:
from odps import DataFrame

data = pd.DataFrame()
item_id_final = []
sim_items_final = []
for i in range(len(item_id)):
    item_id_final.append(item_id[i])
    sim_items_final.append(','.join(sim_items[i]))
data['benefit_id'] = item_id_final
data['sim_items'] = sim_items_final
topai(1, DataFrame(data))